# Optimizing EEG Electrode Use for Workload Detection in Noisy and Silent Environments
Course: Brain Pattern Recognition (03-IMVP-GME), Summer Term 2025

**Authors (Name, E-Mail, Matriculation Number):**
- Julius Harnack, jharnack@uni-bremen.de, 6035044
- Rami Wahed, rami1@uni-bremen.de, 4117690
- Moritz Schäfer, mschaefe@uni-bremen.de, 6128374
- Jannik Wette, jwette@uni-bremen.de, 6137661

**Contributions:**
- Julius Harnack: Data Pipeline, Deep Learning
- Rami Wahed: Data Pipeline, Deep Learning
- Moritz Schäfer: Data Pipeline, Feature-based ML
- Jannik Wette: Data Pipeline, Conclusion and Outlook, Design of the Posters

Everyone was actively involved in the group discussions and the writing of the final report.

## Abstract

The N-Back task is a psychological test in which participants are presented with a sequence of stimuli and must decide if the current stimulus matches the one shown N steps earlier. The task is commonly used to assess working memory and cognitive load.

In this study, we explored how reducing the number of EEG electrodes affects workload detection during a 3-level N-Back task performed in both silent and noisy environments. We collected EEG data from participants using eight electrodes and analyzed the importance of each electrode for classification using a Random Forest (RF) and a Deep Neural Network (DNN).

Our findings indicate that it is possible to maintain reliable workload classification with fewer electrodes, which has implications for the development of more practical and user-friendly Brain-Computer Interfaces (BCIs) for real-world applications. We also found an interesting bias toward the left brain areas, which we believe is due to the hand used by the participants during the task.

## Introduction

The task of this 10-day block course was to develop an end-to-end BCI using modern software libraries and to evaluate it with data collected during a scientific experiment. The experiment involved the N-Back task, a common method for measuring working memory and cognitive load. The task was performed in two different environments: a silent indoor setting and a noisy outdoor setting. Each group was required to implement a BCI and focus on a specific research question.

Our group focused on how reducing the number of EEG electrodes affects workload detection. Specifically, we aimed to answer the following research question:
> To what extent can reliable workload classification be maintained outside the lab using 4 EEG electrodes compared to a baseline of 8 electrodes?

To answer this question, we implemented a complete BCI pipeline using the [MNE](https://mne.tools/stable/index.html) library for EEG data processing, the [scikit-learn](https://scikit-learn.org/stable/) library for machine learning algorithms, and [PyTorch](https://pytorch.org) along with [Braindecode](https://braindecode.org/stable/index.html) for deep learning.

The following sections describe the different steps of the BCI pipeline, including data loading, preprocessing, epoching, and feature extraction. Afterward, the feature-based and deep learning approaches will be discussed. Finally, we will present and discuss the results.

## Prerequisite

Make sure to install all dependencies described in the `README.md`.

In [ ]:
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple
import sys
import os
import random
import numpy as np

# Set random seeds for reproducibility
seed = 42
os.environ["PYTHONHASHSEED"] = str(seed)
random.seed(seed)
np.random.seed(seed)

INDOOR_RAW_FIF_NAME = "indoor_processed_raw.fif"
INDOOR_EPO_FIF_NAME = "indoor_processed-epo.fif"
OUTDOOR_RAW_FIF_NAME = "outdoor_processed_raw.fif"
OUTDOOR_EPO_FIF_NAME = "outdoor_processed-epo.fif"

root_dir = Path.cwd()
data_dir = root_dir / "data"
print(f"Python version: {sys.version}")
print(f"Data folder located at: {data_dir.resolve()}")
assert data_dir.exists(), f"Data folder not found at {data_dir.resolve()}"


@dataclass
class DataLoadingConfig:
    channels_keep: Optional[List[str]] = None
    montage: str = "standard_1020"
    auto_scale_to_volts: bool = True
    max_channels: int = 8


@dataclass
class PreprocessingConfig:
    notch_freq: float = 50.0
    l_freq: float = 1.0
    h_freq: float = 40.0
    reference: str = "average"


@dataclass
class EpochingConfig:
    tmin: float = 0.0
    tmax: Optional[float] = None
    baseline: Optional[Tuple[float, float]] = None
    picks: Optional[List[str]] = None
    reject: Optional[Dict[str, float]] = None


@dataclass
class PipelineConfig:
    data_loading: DataLoadingConfig
    preprocessing: PreprocessingConfig
    epoching: Optional[EpochingConfig] = None
    output_dir: Optional[Path] = None

## Dataset

In this section, we will load the dataset and analyze its structure. The dataset consists of EEG recordings from multiple participants, each with two sessions: one indoor and one outdoor. Each session is stored in an XDF file, which contains both EEG data and event markers.

Some sessions were corrupted and could not be loaded with the `pyxdf` library, so they had to be excluded from further analysis.

In [ ]:
import mne
import pyxdf
import pandas as pd

config = PipelineConfig(
    data_loading=DataLoadingConfig(
        channels_keep=None,
        max_channels=8,
        montage="standard_1020",
    ),
    preprocessing=PreprocessingConfig(l_freq=4.0, h_freq=40.0, notch_freq=50.0),
    epoching=EpochingConfig(),
    output_dir=root_dir / "results",
)


@dataclass
class SessionData:
    participant_name: str
    indoor_session: Optional[mne.io.Raw] = None
    indoor_markers: Optional[pd.DataFrame] = None
    indoor_epochs: Optional[mne.Epochs] = None
    outdoor_session: Optional[mne.io.Raw] = None
    outdoor_markers: Optional[pd.DataFrame] = None
    outdoor_epochs: Optional[mne.Epochs] = None


def load_xdf_safe(path: Path) -> Tuple[Optional[list], Optional[dict]]:
    try:
        streams, header = pyxdf.load_xdf(str(path))
        return streams, header
    except Exception as e:
        print(f"[WARN] XDF-Load failed for {path}: {e}")
        return None, None


def _safe_get(d: dict, key: str, default):
    try:
        v = d.get(key, default)
        if isinstance(v, (list, tuple)) and len(v) == 1:
            return v[0]
        return v
    except Exception:
        return default


def pick_streams(streams: list) -> Tuple[Optional[dict], Optional[dict]]:
    eeg_stream, marker_stream = None, None

    for st in streams:
        info = st.get("info", {})
        stype = str(_safe_get(info, "type", "")).lower()
        sname = str(_safe_get(info, "name", "")).lower()
        ch_n = int(float(_safe_get(info, "channel_count", "0")))

        if ("eeg" in stype or "unicorn" in sname) and ch_n >= 1:
            eeg_stream = st
        if "marker" in stype or "markers" in sname:
            marker_stream = st

    return eeg_stream, marker_stream


def eeg_stream_to_raw(eeg_stream: dict, config: DataLoadingConfig) -> mne.io.Raw:
    info = eeg_stream["info"]
    fs = float(_safe_get(info, "nominal_srate", "0"))
    data = np.array(eeg_stream["time_series"], dtype=float).T

    if config.auto_scale_to_volts:
        med_abs = float(np.nanmedian(np.abs(data)))
        if med_abs > 1e-3:
            print(f"[INFO] Skaliere von µV zu V (median={med_abs:.1f})")
            data *= 1e-6

    # Channel mapping
    channel_mapping = {
        1: "Fz",
        2: "C4",
        3: "Cz",
        4: "C3",
        5: "Pz",
        6: "PO8",
        7: "Oz",
        8: "PO7",
    }
    ch_names = [channel_mapping.get(i + 1, f"EEG{i + 1}") for i in range(data.shape[0])]

    raw = mne.io.RawArray(data, mne.create_info(ch_names, fs, ch_types="eeg"))

    if config.channels_keep:
        keep = [ch for ch in config.channels_keep if ch in raw.ch_names]
    else:
        keep = raw.ch_names[: config.max_channels]
    raw.pick_channels(keep)

    if config.montage:
        try:
            raw.set_montage(config.montage, on_missing="ignore")
        except Exception as e:  # pragma: no cover - GUI/IO bedingt
            print(f"[WARN] Montage fehlgeschlagen: {e}")

    return raw


def get_session_paths(
        experiment_sessions: List[Path],
) -> Tuple[Optional[Path], Optional[Path]]:
    sess01_path: Optional[Path] = None
    sess02_path: Optional[Path] = None

    for session in experiment_sessions:
        if session.parent.parent.name == "ses-S001":
            sess01_path = session
        elif session.parent.parent.name == "ses-S002":
            sess02_path = session
        else:
            print(f"Unbekannter Pfad: {session}")

    return sess01_path, sess02_path


def load_session_data(
        session_path: Optional[Path],
        config: DataLoadingConfig,
) -> Tuple[Optional[mne.io.Raw], Optional[pd.DataFrame]]:
    if session_path is None:
        return None, None

    print(f"Load Session Data: {session_path}")

    streams, _header = load_xdf_safe(session_path)
    if not streams:
        return None, None

    # Stream-Selektion (EEG + Marker)
    eeg_stream, marker_stream = pick_streams(streams)
    if eeg_stream is None:
        print("[WARN] Kein EEG-Stream gefunden")
        return None, None
    first_eeg_timestamp = (
        eeg_stream["time_stamps"][0] if "time_stamps" in eeg_stream else 0
    )
    raw = eeg_stream_to_raw(eeg_stream, config)

    markers = None
    if marker_stream and "time_series" in marker_stream:
        marker_data = np.array(marker_stream["time_series"])
        if marker_data.ndim == 1:
            marker_data = marker_data[:, np.newaxis]
        timestamps = np.array(marker_stream["time_stamps"])
        timestamps = timestamps - first_eeg_timestamp
        markers = pd.DataFrame(
            marker_data, columns=[f"Marker{i + 1}" for i in range(marker_data.shape[1])]
        )
        markers.insert(0, "Timestamp", timestamps)
    return raw, markers


def load_single_session(experiment_dir: Path, config: DataLoadingConfig = None) -> SessionData:
    participant_name = experiment_dir.name.split("_")[-1]
    experiment_sessions = list(experiment_dir.rglob("*.xdf"))
    assert len(experiment_sessions) <= 2, (
        "Mehr als zwei Sessions gefunden - Fehler im Datensatz!"
    )

    indoor_path, outdoor_path = get_session_paths(experiment_sessions)

    indoor_session, indoor_markers = None, None
    try:
        indoor_session, indoor_markers = load_session_data(indoor_path, config)
    except Exception as e:
        print(
            f"[WARN] Fehler beim Laden der Indoor-Session für {participant_name}: {e}"
        )

    outdoor_session, outdoor_markers = None, None
    try:
        outdoor_session, outdoor_markers = load_session_data(outdoor_path, config)
    except Exception as e:
        print(
            f"[WARN] Fehler beim Laden der Outdoor-Session für {participant_name}: {e}"
        )

    return SessionData(
        participant_name=participant_name,
        indoor_session=indoor_session,
        indoor_markers=indoor_markers,
        outdoor_session=outdoor_session,
        outdoor_markers=outdoor_markers,
    )


def load_all_sessions(
        data_dir: Path, config: DataLoadingConfig = None
) -> List[SessionData]:
    experiment_dirs = [p for p in data_dir.resolve().iterdir() if p.is_dir()]
    sessions: List[SessionData] = []

    for experiment_dir in experiment_dirs:
        try:
            session_data = load_single_session(experiment_dir, config)
            sessions.append(session_data)
            print(f"✓ Session geladen: {session_data.participant_name}")
        except Exception as e:  # pragma: no cover
            print(f"✗ Fehler beim Laden {experiment_dir.name}: {e}")

    return sessions


sessions = load_all_sessions(data_dir, config.data_loading)

In [ ]:
# Analyze loaded sessions
indoor_succ, indoor_fail = [], []
outdoor_succ, outdoor_fail = [], []
for session in sessions:
    if session.indoor_session is not None:
        indoor_succ.append(session)
    else:
        indoor_fail.append(session)
    if session.outdoor_session is not None:
        outdoor_succ.append(session)
    else:
        outdoor_fail.append(session)

successful_duration = sum(
    s.indoor_session.n_times / s.indoor_session.info['sfreq'] + s.outdoor_session.n_times / s.outdoor_session.info[
        'sfreq']
    for s in sessions if s.indoor_session is not None and s.outdoor_session is not None
)

total_sessions = sum(1 for s in indoor_succ + indoor_fail + outdoor_succ + outdoor_fail)
metric_dict = {
    "Participants (successfully loaded):": len(sessions),
    "Total Sessions:": total_sessions,
    "Successful Loaded Sessions:": sum(1 for s in indoor_succ + outdoor_succ),
    "Failed Loads:": sum(1 for s in indoor_fail + outdoor_fail),
    "Successful Indoor-Sessions:": f"{sum(1 for s in indoor_succ)}/{sum(1 for s in indoor_succ + indoor_fail)} ({len(indoor_succ) / (len(indoor_succ) + len(indoor_fail)) * 100:.1f}%)",
    "Successful Outdoor-Sessions:": f"{sum(1 for s in outdoor_succ)}/{sum(1 for s in outdoor_succ + outdoor_fail)} ({len(outdoor_succ) / (len(outdoor_succ) + len(outdoor_fail)) * 100:.1f}%)",
    "Failed Sessions:": [(s.participant_name, s.indoor_session, s.outdoor_session) for s in indoor_fail + outdoor_fail
                         if s.indoor_session is None or s.outdoor_session is None],
    "Total Duration (successful):": f"{successful_duration / 60:.1f} min ({successful_duration / 3600:.2f} h)",
}

col1_width = 40
col2_width = 40

print(f"{'Metric':<{col1_width}}{'Value':<{col2_width}}")
print("-" * (col1_width + col2_width))

for key, value in metric_dict.items():
    if isinstance(value, list):
        if value:
            first_item_str = f"- {value[0]}"
            print(f"{key:<{col1_width}}{first_item_str:<{col2_width}}")

            for item in value[1:]:
                item_str = f"- {item}"
                print(f"{'':<{col1_width}}{item_str:<{col2_width}}")
        else:
            print(f"{key:<{col1_width}}")
    else:
        print(f"{key:<{col1_width}}{str(value):<{col2_width}}")

## Preprocessing

The raw EEG data is preprocessed using a notch filter to remove power line noise, re-referencing to the average of all channels and a bandpass filter to retain relevant frequency components.

![Visualization of Preprocessing](images/viz_preprocessing.png)

In [ ]:
def apply_notch_filter(raw: mne.io.Raw, notch_freq: float = 50.0) -> mne.io.Raw:
    raw.notch_filter(
        freqs=[notch_freq, 2 * notch_freq],
        picks="eeg",
        verbose="WARNING",
    )
    return raw


def apply_bandpass_filter(raw: mne.io.Raw, l_freq: float = 1.0, h_freq: float = 40.0) -> mne.io.Raw:
    raw.filter(
        l_freq=l_freq,
        h_freq=h_freq,
        picks="eeg",
        method="fir",
        phase="zero",
        fir_window="hamming",
        verbose="WARNING",
    )
    return raw


def apply_rereferencing(raw: mne.io.Raw, reference: str = "average") -> mne.io.Raw:
    raw.set_eeg_reference(reference)
    return raw


def preprocess_raw(raw: mne.io.Raw, config: PreprocessingConfig) -> mne.io.Raw:
    raw = apply_notch_filter(raw, config.notch_freq)
    raw = apply_bandpass_filter(raw, config.l_freq, config.h_freq)
    raw = apply_rereferencing(raw, config.reference)
    return raw


for session in sessions:
    if session.indoor_session:
        session.indoor_session = preprocess_raw(
            session.indoor_session, config.preprocessing,
        )
    if session.outdoor_session:
        session.outdoor_session = preprocess_raw(
            session.outdoor_session, config.preprocessing
        )

## Marker Annotation

In this section, we combine both the raw data with the marker information and create a combined data structure with the MNE library.

![Visualization of the Markers](images/viz_marker.png)

In [ ]:
def extract_nblock(sequence: List[str], targets: List[int], zero_flag: bool) -> int:
    for name, arg in {"sequence": sequence, "targets": targets}.items():
        if not isinstance(arg, list):
            raise TypeError(f"'{name}' is not a list (got: {type(arg).__name__})")

    if zero_flag:
        return 0

    n_vals = np.zeros(4)
    for t in targets:
        if t >= len(sequence):
            continue
        target_letter = sequence[t]
        if t >= 1 and target_letter == sequence[t - 1]:
            n_vals[1] += 1
        if t >= 2 and target_letter == sequence[t - 2]:
            n_vals[2] += 1
        if t >= 3 and target_letter == sequence[t - 3]:
            n_vals[3] += 1
    return int(np.argmax(n_vals))


def calculate_nvals(df: pd.DataFrame) -> List[int]:
    df = df.copy()
    df["prev_marker"] = df["marker"].shift(1)
    df["prev_prev_marker"] = df["marker"].shift(2)

    mask_seq = df["marker"].str.startswith("sequence") & df["prev_marker"].str.contains(
        "main_block.*start", na=False
    )

    seq_df = (
        df.loc[mask_seq, "marker"]
        .str.removeprefix("sequence_")
        .str.split(",")
        .to_frame(name="sequence")
        .reset_index(drop=True)
    )

    mask_trg = df["marker"].str.startswith("targets") & df[
        "prev_marker"
    ].str.startswith("sequence")

    trg_df = (
        df.loc[mask_trg, "marker"]
        .str.removeprefix("targets_")
        .str.split(",")
        .apply(
            lambda x: [int(i) for i in x] if len(x) > 0 and x[0] != "" else []
        )
        .to_frame(name="targets")
        .reset_index(drop=True)
    )

    n_vals = []
    min_len = min(len(seq_df), len(trg_df))

    for idx in range(min_len):
        seq = seq_df.at[idx, "sequence"]
        trg = trg_df.at[idx, "targets"]
        if idx == 0:
            n_vals.append(extract_nblock(seq, trg, True))
        else:
            n_vals.append(extract_nblock(seq, trg, False))

    return n_vals


def extract_baseline_info(markers_df: pd.DataFrame) -> List[Tuple[float, float, str]]:
    if markers_df is None or markers_df.empty:
        return []

    markers_work = markers_df.copy()
    if "Marker1" in markers_work.columns:
        markers_work = markers_work.rename(columns={"Marker1": "marker"})
    elif "marker" not in markers_work.columns:
        return []

    baseline_info = []

    baseline_starts = markers_work[markers_work["marker"] == "baseline_start"]
    baseline_ends = markers_work[markers_work["marker"] == "baseline_end"]

    for i, (_, start_row) in enumerate(baseline_starts.iterrows()):
        start_time = start_row["Timestamp"]

        subsequent_ends = baseline_ends[baseline_ends["Timestamp"] > start_time]
        if not subsequent_ends.empty:
            end_time = subsequent_ends.iloc[0]["Timestamp"]
            baseline_info.append((start_time, end_time, f"baseline_{i + 1}"))
        else:
            end_time = start_time + 120.0
            baseline_info.append((start_time, end_time, f"baseline_{i + 1}"))

    return baseline_info


def extract_block_info(markers_df: pd.DataFrame) -> List[Tuple[float, float, int, int]]:
    if markers_df is None or markers_df.empty:
        return []

    markers_work = markers_df.copy()
    if "Marker1" in markers_work.columns:
        markers_work = markers_work.rename(columns={"Marker1": "marker"})
    elif "marker" not in markers_work.columns:
        print("Keine 'marker' oder 'Marker1' Spalte gefunden")
        return []

    block_starts = markers_work[
        markers_work["marker"].str.contains("main_block.*start", na=False)
    ]

    if block_starts.empty:
        print("Keine main_block_X_start Marker gefunden")
        return []

    try:
        n_vals = calculate_nvals(markers_work)
    except Exception as e:
        print(f"Fehler bei n-back Berechnung: {e}")
        import traceback

        traceback.print_exc()
        n_vals = [0] * len(block_starts)

    block_info = []

    for idx, (_, row) in enumerate(block_starts.iterrows()):
        start_time = row["Timestamp"]

        if idx + 1 < len(block_starts):
            next_start = block_starts.iloc[idx + 1]["Timestamp"]
            end_time = next_start
        else:
            end_time = markers_work["Timestamp"].max()
            if end_time == start_time:
                end_time = start_time + 60.0

        marker_text = row["marker"]
        try:
            import re

            match = re.search(r"main_block_(\d+)_start", marker_text)
            if match:
                block_num = int(match.group(1))
            else:
                block_num = idx
        except (ValueError, IndexError):
            block_num = idx

        n_back = n_vals[idx] if idx < len(n_vals) else 0

        block_info.append((start_time, end_time, block_num, n_back))

    return block_info


def create_annotations_from_blocks_and_baseline(
        block_info: List[Tuple[float, float, int, int]],
        baseline_info: List[Tuple[float, float, str]],
        eeg_start_time: float,
        sampling_rate: float = None,
) -> mne.Annotations:
    if not block_info and not baseline_info:
        return mne.Annotations(onset=[], duration=[], description=[])

    onsets = []
    durations = []
    descriptions = []

    for start_time, end_time, description in baseline_info:
        onset = start_time - eeg_start_time
        duration = end_time - start_time

        desc = f"baseline"

        onsets.append(onset)
        durations.append(duration)
        descriptions.append(desc)

    for start_time, end_time, block_num, n_back in block_info:
        onset = start_time - eeg_start_time
        duration = end_time - start_time

        desc = f"{n_back}-back"

        onsets.append(onset)
        durations.append(duration)
        descriptions.append(desc)

    return mne.Annotations(
        onset=onsets, duration=durations, description=descriptions, orig_time=None
    )


def annotate_raw_with_markers(raw: mne.io.Raw, markers_df: Optional[pd.DataFrame]) -> mne.io.Raw:
    if markers_df is None or markers_df.empty:
        print("No marker data available!")
        return raw

    block_info = extract_block_info(markers_df)

    baseline_info = extract_baseline_info(markers_df)

    if not block_info and not baseline_info:
        print("No valid block or baseline info extracted!")
        return raw

    annotations = create_annotations_from_blocks_and_baseline(
        block_info, baseline_info, 0.0
    )

    raw.set_annotations(annotations)

    return raw


for session in sessions:
    if session.indoor_session and session.indoor_markers is not None:
        session.indoor_session = annotate_raw_with_markers(
            session.indoor_session, session.indoor_markers
        )
    if session.outdoor_session and session.outdoor_markers is not None:
        session.outdoor_session = annotate_raw_with_markers(
            session.outdoor_session, session.outdoor_markers
        )

## Epoching

We segment the continuous EEG data into 4-second windows with 50% overlap to prepare it for feature extraction and classification. This approach also allows the algorithms to incorporate more temporal context.

During the block course, we used this method as a sequential step in a pipeline, ensuring that all subsequent experiments (for both feature-based and deep learning approaches) were run on the same epochs. This process was error-prone and caused data leakage because a lot of information from the overlap went directly into the test set with standard k-fold validation approaches. This resulted in inflated validation accuracies. We have since restructured the code to ensure that epochs are created during the evaluation process.

![Visualization of the Epoching](images/viz_epoching.png)

In [ ]:
def create_epochs_from_raw(
        raw: mne.io.Raw, config: EpochingConfig = None
) -> Optional[mne.Epochs]:
    if not raw.annotations or len(raw.annotations) == 0:
        print("No annotation found in raw data!")
        return None

    segment_length = 4.0
    overlap = 2.0

    all_events = []
    event_id = {
        'baseline': 0,
        '0-back': 1,
        '1-back': 2,
        '2-back': 3,
        '3-back': 4
    }

    for block_idx, annot in enumerate(raw.annotations):
        description = annot['description']

        start_time = annot['onset']
        duration = annot['duration']

        block_events = mne.make_fixed_length_events(
            raw,
            id=event_id[description],
            start=start_time,
            stop=start_time + duration,
            duration=segment_length,
            overlap=overlap
        )

        if len(block_events) == 0:
            print(f"No segments for block {block_idx} (too short)")
            continue

        print(f"{len(block_events)} Segmente a {segment_length}s")
        all_events.append(block_events)

    if not all_events:
        print("No valid events created!")
        return None

    events = np.vstack(all_events)
    events = events[events[:, 0].argsort()]

    print(f"\nCreate {len(events)} epochs")
    print(f"Event IDs: {event_id}")

    try:
        epochs = mne.Epochs(
            raw,
            events=events,
            event_id=event_id,
            tmin=config.tmin,
            tmax=config.tmax or segment_length,
            baseline=config.baseline,
            picks=config.picks,
            reject=config.reject,
            preload=True,
            verbose=False
        )

        data_shape = epochs.get_data().shape
        print(f"{len(epochs)} epochs created")
        print(f"Shape: {data_shape}")

        # Several assertions to validate the epochs
        assert epochs is not None
        assert len(epochs) > 0

        assert len(data_shape) == 3
        assert data_shape[0] > 0
        assert data_shape[1] > 0
        assert data_shape[2] > 0

        assert hasattr(epochs, 'event_id')
        assert len(epochs.event_id) > 0
        assert epochs.info['sfreq'] > 0

        onset_times = epochs.events[:, 0] / epochs.info['sfreq']
        assert np.all(onset_times[:-1] <= onset_times[1:])

        return epochs

    except Exception as e:
        print(f"[ERROR] Fehler beim Epoching: {e}")
        import traceback
        traceback.print_exc()
        return None


for session in sessions:
    if session.participant_name == "jannik":
        continue

    if session.indoor_session and session.indoor_session.annotations:
        session.indoor_epochs = create_epochs_from_raw(
            session.indoor_session.copy(), config.epoching
        )
        if session.indoor_epochs:
            print(f"Indoor epochs created for {session.participant_name}: {len(session.indoor_epochs)} epochs")

    if session.outdoor_session and session.outdoor_session.annotations:
        session.outdoor_epochs = create_epochs_from_raw(
            session.outdoor_session.copy(), config.epoching
        )
        if session.outdoor_epochs:
            print(f"Outdoor epochs created for {session.participant_name}: {len(session.outdoor_epochs)} epochs")

## Saving Preprocessed Data

Both the preprocessed raw data and the created epochs are saved in dedicated FIF files for each participant. This allows for easy reuse of the data in future experiments without needing to repeat the preprocessing and epoching steps.

In [ ]:
def save_processed_data() -> None:
    assert config.output_dir is not None
    config.output_dir.mkdir(exist_ok=True)

    for session in sessions:
        session_dir = config.output_dir / session.participant_name
        session_dir.mkdir(exist_ok=True)

        print(f"Saving {session.participant_name}:")

        if session.indoor_session:
            indoor_path = session_dir / INDOOR_RAW_FIF_NAME
            session.indoor_session.save(str(indoor_path), overwrite=True)
            assert indoor_path.exists()
            print(f"Saved indoor session to {indoor_path}")
        else:
            print(f"No indoor session for {session.participant_name} available")

        if session.outdoor_session:
            outdoor_path = session_dir / OUTDOOR_RAW_FIF_NAME
            session.outdoor_session.save(str(outdoor_path), overwrite=True)
            assert outdoor_path.exists()
            print(f"Saved outdoor session to {outdoor_path}")
        else:
            print(f"No outdoor session for {session.participant_name} available")

        if session.indoor_epochs:
            indoor_epochs_path = session_dir / INDOOR_EPO_FIF_NAME
            session.indoor_epochs.save(str(indoor_epochs_path), overwrite=True)

            assert indoor_epochs_path.exists()

            # Verify by reloading
            loaded_epochs = mne.read_epochs(str(indoor_epochs_path), verbose=False)
            assert len(loaded_epochs) == len(session.indoor_epochs)
            assert (loaded_epochs.event_id == session.indoor_epochs.event_id)

            print(f"Saved indoor epochs and verified ({len(loaded_epochs)} epochs.)"
                  )

        if session.outdoor_epochs:
            outdoor_epochs_path = session_dir / OUTDOOR_EPO_FIF_NAME
            session.outdoor_epochs.save(str(outdoor_epochs_path), overwrite=True)

            assert outdoor_epochs_path.exists()

            # Verify by reloading
            loaded_epochs = mne.read_epochs(str(outdoor_epochs_path), verbose=False)
            assert len(loaded_epochs) == len(session.outdoor_epochs)
            assert loaded_epochs.event_id == session.outdoor_epochs.event_id

            print(f"Saved outdoor epochs and verified ({len(loaded_epochs)} epochs.)")

    print(f"Files were successfully saved to {config.output_dir}")


save_processed_data()

After executing the pipeline, there should be a `results` folder containing subfolders for each participant. Each participant folder should contain the preprocessed raw data and the created epochs for both indoor and outdoor sessions (if available). The following script checks for the existence of these files.

In [ ]:
# Verify output after running this notebook
assert config.output_dir.exists()

for session in sessions:
    session_dir = config.output_dir / session.participant_name

    assert session_dir.exists()
    assert any(session_dir.iterdir())

    if session.indoor_session is not None:
        indoor_path = session_dir / INDOOR_RAW_FIF_NAME
        assert indoor_path.exists()

    if session.outdoor_session is not None:
        outdoor_path = session_dir / OUTDOOR_RAW_FIF_NAME
        assert outdoor_path.exists()

    if session.indoor_epochs is not None:
        indoor_epochs_path = session_dir / INDOOR_EPO_FIF_NAME
        assert indoor_epochs_path.exists()

    if session.outdoor_epochs is not None:
        outdoor_epochs_path = session_dir / OUTDOOR_EPO_FIF_NAME
        assert outdoor_epochs_path.exists()

##  Feature-based ML: Random Forest

Results are presented in dedicated notebook: `Final_Submission_Feature-Based_ML.ipynb`.

## Deep Learning

Results are presented in dedicated notebook: `Final_Submission_Deep_Learning.ipynb`.

## Conclusion and Outlook

In this pilot study, we investigated the ability of two machine learning models (Random Forest and a Deep Neural Network) to classify data recorded with an 8 electrode EEG device into 3 different states of mental load - depending on the difficulty level of the given NBack-Task (1-, 2- and 3-Back). The goal was to determine how far the electrode count can be reduced while still preserving adequate accuracy of classification. 

Using the Random Forest Model, we identified first that only 5 electrodes (Oz, C3, PO7, C4 and Fz) were relevant for classification accuracy. Further comparison of the mean accuracy over all subjects for all 8 electrodes (0.593) vs. the mean accuracy for only the top 4 most relevant electrodes (0.573) showed only a minor drop in accuracy (0.02). 

A similar finding was seen using the Deep Learning Model "AttentionBaseNet" from braindecode. Here also a difference between electrodes in terms of relevance for classification was seen. PO7, C3 and Oz are the most relevant which show a >0.3 drop in accuracy. The same electrodes as in the random forest model. Also shortly followed by C4 and Fz which are closely below 0.3 in accuracy drop. The model results themselves where not as clear because of overfitting, caused by the the data leakage introduced by the nature of our epoch segmentation (50% overlap between segments).

Another issue in both models was to classify data from recordings outside the lab, with the model trained only on inside the lab recorded data. To improve here, one could include outside data in the training or consider selecting electrodes that are relevant for the outside data instead of using indoor data to sort for electrode relevance. 

Our results indicate that reduction of electrode amount far more than usual, could be a feasible option especially under consideration of the development of real-time BCIs. With reduction in electrodes, the handling and preparation improves as well as the ability of everyday wear of such devices. 

Future studies building on this pilot project should investigate further drop of electrodes past 4 and consider a step by step approach - Showing accuracies from an 8 electrode model down to a 1 electrode model in steps of 1 electrode dropped each. This will help pinpoint a potential sweetspot. Also more data could be recorded and used as well as more consideration of meta data e.g. left-handed vs right-handed to see if potential eeg patterns are traceable to such differences in subjects.



## Appendix

### AI Usage Declaration

The following AI tools were used during the block course:
- **Github Copilot (Claude Sonnet 4):**
    - The model was instructed to visualize certain aspects of the data. For example, the model was asked to write a Python script to output a
- **Github Copilot (multiple models):**
    - The models were used to solve issues related to error messages.
- **Gemini 2.5 Pro:**
    - Deep Research. For example, the model was asked to find relevant features in similar settings from multiple scientific sources.
    - The model was used to enhance the writing style of this notebook. For example, an already in the English language formulated sentence was enhanced by using more academic vocabulary.
- **DeepL:** Translation from German to English.